In [1]:
import numpy as np
import matplotlib.pyplot as plt
from lm_polygraph.ue_metrics.pred_rej_area import PredictionRejectionArea
from lm_polygraph.ue_metrics.ue_metric import (
    get_random_scores,
    normalize_metric,
)
import sklearn
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from collections import defaultdict
from sacrebleu import CHRF, BLEU
from utils import extract_and_prepare_data

/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [2]:
methods_dict = {
    'MaximumSequenceProbability': 'MSP',
    'Perplexity': 'PPL',
    'MeanTokenEntropy': 'MTE',
    'MonteCarloSequenceEntropy': 'MCSE',
    'MonteCarloNormalizedSequenceEntropy': 'MCNSE',
}

DATASETS = [
    'wmt14_csen',
    'wmt14_ruen',
]

all_metrics = ['Comet', 'BLEU']

In [36]:
import pandas as pd
import numpy as np
from sklearn import datasets, linear_model
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
from scipy import stats


In [31]:
import numpy as np
import sklearn.linear_model
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import t

bootstrap_results = {}
coefficients = []  # List to store x1 coefficient results
coefficients_test = []
for dataset in DATASETS:
    train_ue_values, \
    test_ue_values, \
    train_metric_values, \
    test_metric_values, \
    train_gen_lengths, \
    test_gen_lengths = extract_and_prepare_data(dataset, methods_dict, all_metrics, model='llama1b')

    # Filter based on quantiles for train and test separately
    upper_q_train = np.quantile(train_gen_lengths, 0.95)
    lower_q_train = np.quantile(train_gen_lengths, 0.05)
    below_q_train = (train_gen_lengths < upper_q_train) & (train_gen_lengths > lower_q_train)
    train_gen_lengths = train_gen_lengths[below_q_train]

    upper_q_test = np.quantile(test_gen_lengths, 0.95)
    lower_q_test = np.quantile(test_gen_lengths, 0.05)
    below_q_test = (test_gen_lengths < upper_q_test) & (test_gen_lengths > lower_q_test)
    test_gen_lengths = test_gen_lengths[below_q_test]

    dataset_results = {}

    for metric in all_metrics:  # Iterate over actual metric names
        try:
            # Normalize sequence lengths
            gen_length_scaler_train = MinMaxScaler()
            train_gen_lengths_normalized = gen_length_scaler_train.fit_transform(train_gen_lengths[:, np.newaxis]).squeeze()

            gen_length_scaler_test = MinMaxScaler()
            test_gen_lengths_normalized = gen_length_scaler_test.fit_transform(test_gen_lengths[:, np.newaxis]).squeeze()

            # Normalize metric values
            train_metric = train_metric_values[metric][below_q_train]
            scaler_train = MinMaxScaler()
            train_normalized_metric_values = scaler_train.fit_transform(train_metric[:, np.newaxis]).squeeze()

            test_metric = test_metric_values[metric][below_q_test]
            scaler_test = MinMaxScaler()
            test_normalized_metric_values = scaler_test.fit_transform(test_metric[:, np.newaxis]).squeeze()
            X = train_gen_lengths_normalized
            y = train_normalized_metric_values

            X2 = sm.add_constant(X)

            # Fit the OLS model
            est = sm.OLS(y, X2)
            est2 = est.fit()

            # Print the summary of the model
            summary = est2.summary2().tables[1]

            # Print the row for the coefficients and related statistics
            x1_values = summary.loc['x1']
            # print(x1_values.index)
            coefficient = x1_values['Coef.']
            # print(coefficient)
            # Std.Err.', 't', 'P>|t|',
            std_error = x1_values['Std.Err.']
            t_value = x1_values['t']
            p_value = x1_values['P>|t|']
            conf_int_low = x1_values['[0.025']
            conf_int_high = x1_values['0.975]']

            # Store the x1 coefficient row values in the coefficients list
            coefficients.append({
                'dataset': dataset,
                'metric': metric,
                'x1_coef': coefficient,
                'x1_std_err': std_error,
                'x1_t_value': t_value,
                'x1_p_value': p_value,
                'x1_conf_int_low': conf_int_low,
                'x1_conf_int_high': conf_int_high
            })

            X = test_gen_lengths_normalized
            y = test_normalized_metric_values

            X2 = sm.add_constant(X)

            # Fit the OLS model
            est = sm.OLS(y, X2)
            est2 = est.fit()

            # Print the summary of the model
            summary = est2.summary2().tables[1]

            # Print the row for the coefficients and related statistics
            x1_values = summary.loc['x1']
            # print(x1_values.index)
            coefficient = x1_values['Coef.']
            # print(coefficient)
            # Std.Err.', 't', 'P>|t|',
            std_error = x1_values['Std.Err.']
            t_value = x1_values['t']
            p_value = x1_values['P>|t|']
            conf_int_low = x1_values['[0.025']
            conf_int_high = x1_values['0.975]']

            # Store the x1 coefficient row values in the coefficients list
            coefficients_test.append({
                'dataset': dataset,
                'metric': metric,
                'x1_coef': coefficient,
                'x1_std_err': std_error,
                'x1_t_value': t_value,
                'x1_p_value': p_value,
                'x1_conf_int_low': conf_int_low,
                'x1_conf_int_high': conf_int_high
            })

            # dataset_results[metric] = summary

        except Exception as ex:
            print(f"Exception: {ex}\n Metric: {metric}, Train Lengths: {len(train_gen_lengths)}, Test Lengths: {len(test_gen_lengths)}")

    # bootstrap_results[dataset] = dataset_results


Loading NLI model...
/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Initializing stat calculators...
Initializing InitialStateCalculator
Initializing SemanticMatrixCalculator
Initializing SemanticClassesCalculator
Initializing 

Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7fecf063b310>]


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Initializing stat calculators...
Initializing InitialStateCalculator
Initializing SemanticMatrixCalculator
Initializing SemanticClassesCalculator
Initializing GreedyProbsCalculator
Initializing EntropyCalculator
Initializing GreedyLMProbsCalculator
Initializing SamplingGenerationCalculator
Initializing BartScoreCalculator
Initializing ModelScoreCalculat

Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7fece82a0250>]


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Initializing stat calculators...
Initializing InitialStateCalculator
Initializing SemanticMatrixCalculator
Initializing SemanticClassesCalculator
Initializing GreedyProbsCalculator
Initializing EntropyCalculator
Initializing GreedyLMProbsCalculator
Initializing SamplingGenerationCalculator
Initializing BartScoreCalculator
Initializing ModelScoreCalculat

Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7fecf0a2dfc0>]


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Initializing stat calculators...
Initializing InitialStateCalculator
Initializing SemanticMatrixCalculator
Initializing SemanticClassesCalculator
Initializing GreedyProbsCalculator
Initializing EntropyCalculator
Initializing GreedyLMProbsCalculator
Initializing SamplingGenerationCalculator
Initializing BartScoreCalculator
Initializing ModelScoreCalculat

Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7fecf06572e0>]

X1 Coefficients Summary:
Dataset: wmt14_csen, Metric: Comet
  x1 Coefficient: 0.006812
  x1 Std Error: 0.015011
  x1 t-value: 0.453804
  x1 p-value: 0.650026
  x1 Confidence Interval: [-0.022629, 0.036253]
--------------------------------------------------
Dataset: wmt14_csen, Metric: BLEU
  x1 Coefficient: 0.035035
  x1 Std Error: 0.016597
  x1 t-value: 2.110899
  x1 p-value: 0.034922
  x1 Confidence Interval: [0.002483, 0.067587]
--------------------------------------------------
Dataset: wmt14_ruen, Metric: Comet
  x1 Coefficient: 0.008049
  x1 Std Error: 0.016081
  x1 t-value: 0.500540
  x1 p-value: 0.616759
  x1 Confidence Interval: [-0.023492, 0.039590]
--------------------------------------------------
Dataset: wmt14_ruen, Metric: BLEU
  x1 Coefficient: -0.024985
  x1 Std Error: 0.017976
  x1 t-value: -1.389907
  x1 p-value: 0.164736
  x1 Confidence Interval: [-0.0602

In [33]:
print("\nX1 Coefficients Summary:")
for result in coefficients_test:
    print(f"Dataset: {result['dataset']}, Metric: {result['metric']}")
    print(f"  x1 Coefficient: {result['x1_coef']:.6f}")
    print(f"  x1 Std Error: {result['x1_std_err']:.6f}")
    print(f"  x1 t-value: {result['x1_t_value']:.6f}")
    print(f"  x1 p-value: {result['x1_p_value']:.6f}")
    print(f"  x1 Confidence Interval: [{result['x1_conf_int_low']:.6f}, {result['x1_conf_int_high']:.6f}]")
    print("-" * 50)



X1 Coefficients Summary:
Dataset: wmt14_csen, Metric: Comet
  x1 Coefficient: -0.008752
  x1 Std Error: 0.015254
  x1 t-value: -0.573749
  x1 p-value: 0.566212
  x1 Confidence Interval: [-0.038671, 0.021167]
--------------------------------------------------
Dataset: wmt14_csen, Metric: BLEU
  x1 Coefficient: 0.003860
  x1 Std Error: 0.015013
  x1 t-value: 0.257111
  x1 p-value: 0.797124
  x1 Confidence Interval: [-0.025586, 0.033306]
--------------------------------------------------
Dataset: wmt14_ruen, Metric: Comet
  x1 Coefficient: -0.049650
  x1 Std Error: 0.011622
  x1 t-value: -4.272082
  x1 p-value: 0.000020
  x1 Confidence Interval: [-0.072445, -0.026856]
--------------------------------------------------
Dataset: wmt14_ruen, Metric: BLEU
  x1 Coefficient: -0.027114
  x1 Std Error: 0.016581
  x1 t-value: -1.635268
  x1 p-value: 0.102172
  x1 Confidence Interval: [-0.059634, 0.005406]
--------------------------------------------------


In [37]:
import numpy as np
import sklearn.linear_model
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import t

bootstrap_results = {}
coefficients = []  # List to store x1 coefficient results
coefficients_test = []
for dataset in DATASETS:
    train_ue_values, \
    test_ue_values, \
    train_metric_values, \
    test_metric_values, \
    train_gen_lengths, \
    test_gen_lengths = extract_and_prepare_data(dataset, methods_dict, all_metrics, model='llama1b')

    # Filter based on quantiles for train and test separately
    upper_q_train = np.quantile(train_gen_lengths, 0.95)
    lower_q_train = np.quantile(train_gen_lengths, 0.05)
    below_q_train = (train_gen_lengths < upper_q_train) & (train_gen_lengths > lower_q_train)
    train_gen_lengths = train_gen_lengths[below_q_train]

    upper_q_test = np.quantile(test_gen_lengths, 0.95)
    lower_q_test = np.quantile(test_gen_lengths, 0.05)
    below_q_test = (test_gen_lengths < upper_q_test) & (test_gen_lengths > lower_q_test)
    test_gen_lengths = test_gen_lengths[below_q_test]

    dataset_results = {}

    for metric in list(test_ue_values.keys()):  # Iterate over actual metric names
        try:
            # Normalize sequence lengths
            gen_length_scaler_train = MinMaxScaler()
            train_gen_lengths_normalized = gen_length_scaler_train.fit_transform(train_gen_lengths[:, np.newaxis]).squeeze()

            gen_length_scaler_test = MinMaxScaler()
            test_gen_lengths_normalized = gen_length_scaler_test.fit_transform(test_gen_lengths[:, np.newaxis]).squeeze()

            # Normalize metric values
            train_metric = train_ue_values[metric][below_q_train]
            scaler_train = MinMaxScaler()
            train_normalized_metric_values = scaler_train.fit_transform(train_metric[:, np.newaxis]).squeeze()

            test_metric = test_ue_values[metric][below_q_test]
            scaler_test = MinMaxScaler()
            test_normalized_metric_values = scaler_test.fit_transform(test_metric[:, np.newaxis]).squeeze()
            X = train_gen_lengths_normalized
            y = train_normalized_metric_values

            X2 = sm.add_constant(X)

            # Fit the OLS model
            est = sm.OLS(y, X2)
            est2 = est.fit()

            # Print the summary of the model
            summary = est2.summary2().tables[1]

            # Print the row for the coefficients and related statistics
            x1_values = summary.loc['x1']
            # print(x1_values.index)
            coefficient = x1_values['Coef.']
            # print(coefficient)
            # Std.Err.', 't', 'P>|t|',
            std_error = x1_values['Std.Err.']
            t_value = x1_values['t']
            p_value = x1_values['P>|t|']
            conf_int_low = x1_values['[0.025']
            conf_int_high = x1_values['0.975]']

            # Store the x1 coefficient row values in the coefficients list
            coefficients.append({
                'dataset': dataset,
                'metric': metric,
                'x1_coef': coefficient,
                'x1_std_err': std_error,
                'x1_t_value': t_value,
                'x1_p_value': p_value,
                'x1_conf_int_low': conf_int_low,
                'x1_conf_int_high': conf_int_high
            })

            X = test_gen_lengths_normalized
            y = test_normalized_metric_values

            X2 = sm.add_constant(X)

            # Fit the OLS model
            est = sm.OLS(y, X2)
            est2 = est.fit()

            # Print the summary of the model
            summary = est2.summary2().tables[1]

            # Print the row for the coefficients and related statistics
            x1_values = summary.loc['x1']
            # print(x1_values.index)
            coefficient = x1_values['Coef.']
            # print(coefficient)
            # Std.Err.', 't', 'P>|t|',
            std_error = x1_values['Std.Err.']
            t_value = x1_values['t']
            p_value = x1_values['P>|t|']
            conf_int_low = x1_values['[0.025']
            conf_int_high = x1_values['0.975]']

            # Store the x1 coefficient row values in the coefficients list
            coefficients_test.append({
                'dataset': dataset,
                'metric': metric,
                'x1_coef': coefficient,
                'x1_std_err': std_error,
                'x1_t_value': t_value,
                'x1_p_value': p_value,
                'x1_conf_int_low': conf_int_low,
                'x1_conf_int_high': conf_int_high
            })

            # dataset_results[metric] = summary

        except Exception as ex:
            print(f"Exception: {ex}\n Metric: {metric}, Train Lengths: {len(train_gen_lengths)}, Test Lengths: {len(test_gen_lengths)}")



Loading NLI model...
/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Initializing stat calculators...
Initializing InitialStateCalculator
Initializing SemanticMatrixCalculator
Initializing SemanticClassesCalculator
Initializing 

Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7fecf0a5a8f0>]


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Initializing stat calculators...
Initializing InitialStateCalculator
Initializing SemanticMatrixCalculator
Initializing SemanticClassesCalculator
Initializing GreedyProbsCalculator
Initializing EntropyCalculator
Initializing GreedyLMProbsCalculator
Initializing SamplingGenerationCalculator
Initializing BartScoreCalculator
Initializing ModelScoreCalculat

Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7fecf06542b0>]


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Initializing stat calculators...
Initializing InitialStateCalculator
Initializing SemanticMatrixCalculator
Initializing SemanticClassesCalculator
Initializing GreedyProbsCalculator
Initializing EntropyCalculator
Initializing GreedyLMProbsCalculator
Initializing SamplingGenerationCalculator
Initializing BartScoreCalculator
Initializing ModelScoreCalculat

Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7fece175d840>]


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Initializing stat calculators...
Initializing InitialStateCalculator
Initializing SemanticMatrixCalculator
Initializing SemanticClassesCalculator
Initializing GreedyProbsCalculator
Initializing EntropyCalculator
Initializing GreedyLMProbsCalculator
Initializing SamplingGenerationCalculator
Initializing BartScoreCalculator
Initializing ModelScoreCalculat

Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7fece82a0a30>]


In [42]:
# LaTeX table header
latex_table = """
\\begin{table}[ht]
\\centering
\\scalebox{0.7}{
\\begin{tabular}{lllllll}
\\hline
Dataset & UE & x1 Coefficient & x1 Std Error & x1 t-value & x1 p-value & x1 Confidence Interval \\\\
\\hline
"""

# Loop through the list of dictionaries and append each row to the table
for result in coefficients_test:
    latex_table += f"{result['dataset'].replace('_', '-')} & {result['metric']} & {result['x1_coef']:.3f} & {result['x1_std_err']:.3f} & {result['x1_t_value']:.3f} & {result['x1_p_value']:.3f} & [{result['x1_conf_int_low']:.3f}, {result['x1_conf_int_high']:.3f}] \\\\ \n"

# LaTeX table footer
latex_table += """
\\hline
\\end{tabular}}
\\caption{X1 Coefficients Summary}
\\end{table}
"""

# Print the LaTeX table
print(latex_table)



\begin{table}[ht]
\centering
\scalebox{0.7}{
\begin{tabular}{lllllll}
\hline
Dataset & UE & x1 Coefficient & x1 Std Error & x1 t-value & x1 p-value & x1 Confidence Interval \\
\hline
wmt14-csen & MSP & 0.361 & 0.008 & 44.976 & 0.000 & [0.345, 0.377] \\ 
wmt14-csen & PPL & -0.072 & 0.013 & -5.327 & 0.000 & [-0.098, -0.045] \\ 
wmt14-csen & MTE & -0.078 & 0.014 & -5.504 & 0.000 & [-0.106, -0.050] \\ 
wmt14-csen & MCSE & 0.413 & 0.008 & 50.666 & 0.000 & [0.397, 0.429] \\ 
wmt14-csen & MCNSE & 0.019 & 0.014 & 1.329 & 0.184 & [-0.009, 0.047] \\ 
wmt14-ruen & MSP & 0.452 & 0.009 & 50.386 & 0.000 & [0.435, 0.470] \\ 
wmt14-ruen & PPL & -0.071 & 0.013 & -5.677 & 0.000 & [-0.096, -0.047] \\ 
wmt14-ruen & MTE & -0.069 & 0.012 & -5.663 & 0.000 & [-0.093, -0.045] \\ 
wmt14-ruen & MCSE & 0.480 & 0.009 & 52.659 & 0.000 & [0.462, 0.498] \\ 
wmt14-ruen & MCNSE & 0.016 & 0.014 & 1.167 & 0.243 & [-0.011, 0.044] \\ 

\hline
\end{tabular}}
\caption{X1 Coefficients Summary}
\end{table}



In [ ]:

# # Display final results
# for dataset, results in bootstrap_results.items():
#     print(f"\nDataset: {dataset}")
#     for metric, stats in results.items():
#         print(f"  Metric: {metric}")
#         print(f"    Mean Coef Train: {stats['mean_coef_train']:.3f}")
#         print(f"    Mean Coef Test: {stats['mean_coef_test']:.3f}")
#         print(f"    Std Dev Coef Train: {stats['std_coef_train']:.3f}")
#         print(f"    Std Dev Coef Test: {stats['std_coef_test']:.3f}")
#         print(f"    95% CI Train: ({stats['95%_CI_train'][0]:.3f}, {stats['95%_CI_train'][1]:.3f})")
#         print(f"    95% CI Test: ({stats['95%_CI_test'][0]:.3f}, {stats['95%_CI_test'][1]:.3f})")
#         print(f"    p-value Train: {stats['p_value_train']:.10f} {'**' if stats['p_value_train'] < 0.05 else ''}")
#         print(f"    p-value Test: {stats['p_value_test']:.10f} {'**' if stats['p_value_test'] < 0.05 else ''}")
